In [1]:
%cd D:\Study Documents\Other subjects\CoDraft-Bench
CD_PATH = "/kaggle/working/CoDraft-Bench"

D:\Study Documents\Other subjects\CoDraft-Bench


In [2]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *


In [7]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cpu


In [8]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_NAME = "dunzhang/stella_en_400M_v5"
MODEL_TYPE = "siamese"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

In [9]:
PATH_SIMCSE = f"{WORK_DIR}/simcse_model"
PATH_FINAL = f"{WORK_DIR}/final_similarity_model"

In [11]:
tokenizer = get_tokenizer(MODEL_NAME)

In [12]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED,
    rebalance=True
)

Number of new cross_pairing: 134124


Map: 100%|██████████| 8534/8534 [00:11<00:00, 770.85 examples/s]


In [16]:
data_manager._all_texts()
all_texts = data_manager.get_all_texts()


Total unique sentences for SimCSE: 6909


In [17]:
train_simcse(MODEL_NAME, all_texts, PATH_SIMCSE, DEVICE)

STEP 1: Training SimCSE


A new version of the following files was downloaded from https://huggingface.co/dunzhang/stella_en_400M_v5:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/dunzhang/stella_en_400M_v5:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


KeyboardInterrupt: 

In [ ]:
model = SiameseClassifier(PATH_SIMCSE, num_classes=5)
model.to(DEVICE)
new_tokenizer = model.encoder.tokenizer
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED,
    rebalance=True
)

In [ ]:
train_loader, val_loader = data_manager.get_dataloaders(model_type='siamese')

In [ ]:
train_df, val_df, test_df = data_manager.get_data()

In [ ]:
class_weights = compute_class_weight(train_df['Similarity'])

In [ ]:
train_siamese(PATH_SIMCSE, train_loader, val_loader, class_weights, PATH_FINAL, DEVICE)

In [ ]:
train_dataloader, evaluator = data_manager.get_dataloaders(model_type="cross_encoder")